# MuSeg — Fat Fraction Evaluation vs Ground Truth

Computes per-muscle metrics for MuSeg fat-fraction segmentations against myosegmenTUM GT.
Saves one CSV per muscle to `results_fat_frac/`.

For water and Dixon evaluation see `column_compare_museg_all_modalities.ipynb`.

In [ ]:
import glob
import os
import re
import numpy as np
import pandas as pd
import SimpleITK as sitk
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [ ]:
!pwd

In [ ]:
BOUNDARY_DISTANCE = 1

SEG_DIR    = os.path.join('..', '..', 'museg_thigh_segs', 'museg_thigh_segs')
GT_BASE    = os.path.join('..', '..', 'myosegmenTUM')
RESULT_DIR = os.path.join('..', 'results')
os.makedirs(RESULT_DIR, exist_ok=True)

# MuSeg label indices in the output NIfTI
# No L/R distinction — both sides share one label
MUSEG_GRACILIS  = 6
MUSEG_SARTORIUS = 5

# Ground-truth label indices in combined_gt_stack*.mha
GT_L_GRACILIS  = 1
GT_R_GRACILIS  = 5
GT_L_SARTORIUS = 4
GT_R_SARTORIUS = 8

seg_files = sorted(f for f in os.listdir(SEG_DIR) if f.endswith('_museg.nii.gz'))
print(f'Found {len(seg_files)} segmentation files')

## Right Gracilis

In [ ]:
results_r_gracilis = []

for seg_file in seg_files:
    stem    = seg_file.replace('_museg.nii.gz', '')
    subject = stem.split('_FATFRACTION')[0]
    m       = re.search(r'stack(\d+)', stem)
    if not m:
        print(f'  could not parse stack number: {seg_file}, skipping')
        continue
    stack_num = m.group(1)
    gt_name = os.path.join(GT_BASE, subject, 'SegmentationMasks',
                           f'combined_gt_stack{stack_num}.mha')
    print(seg_file)

    gt_image  = sitk.ReadImage(gt_name)
    pred_image = sitk.ReadImage(os.path.join(SEG_DIR, seg_file))

    gt   = sitk.Cast(gt_image == GT_R_GRACILIS, sitk.sitkUInt8)
    gt_arr = sitk.GetArrayFromImage(gt).astype(float)

    pred_arr = sitk.GetArrayFromImage(
        sitk.Cast(pred_image == MUSEG_GRACILIS, sitk.sitkUInt8)).astype(float)
    pred_sitk = sitk.GetImageFromArray(pred_arr.astype(np.uint8))
    pred_sitk.CopyInformation(gt_image)
    pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(gt, pred)

    if gt_arr.sum() > 0 and pred_arr.sum() > 0:
        hd_filter = sitk.HausdorffDistanceImageFilter()
        hd_filter.Execute(gt, pred)
        hd = hd_filter.GetHausdorffDistance()
    else:
        print(f'  empty mask (gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD=NaN')
        hd = np.nan

    results_r_gracilis.append({
        'image':      gt_name,
        'pred_label': seg_file,
        'R_gracilis_lower_dice:':          dice_filter.GetDiceCoefficient(),
        'R_gracilis_lower_Hausdorff:':     hd,
        'R_gracilis_jaccard':              dice_filter.GetJaccardCoefficient(),
        'R_gracilis_volume_similarity':    dice_filter.GetVolumeSimilarity(),
        'R_gracilis_falseNegative':        dice_filter.GetFalseNegativeError(),
        'R_gracilis_falsePostivie':        dice_filter.GetFalsePositiveError(),
        'R_gracilis_binary_cross_entropy': binary_cross_entropy(gt_arr, pred_arr),
        'R_gracilis_boundary_iou_3d':      boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr),
        'R_gracilis_inter_slice_dice_pred': inter_slice_dice(pred_arr),
        'R_gracilis_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
    })

df_r_gracilis = pd.DataFrame(results_r_gracilis)
df_r_gracilis.to_csv(os.path.join(RESULT_DIR, 'df_r_gracilis_museg.csv'))
df_r_gracilis

## Left Gracilis

In [ ]:
results_l_gracilis = []

for seg_file in seg_files:
    stem    = seg_file.replace('_museg.nii.gz', '')
    subject = stem.split('_FATFRACTION')[0]
    m       = re.search(r'stack(\d+)', stem)
    if not m:
        continue
    stack_num = m.group(1)
    gt_name = os.path.join(GT_BASE, subject, 'SegmentationMasks',
                           f'combined_gt_stack{stack_num}.mha')
    print(seg_file)

    gt_image   = sitk.ReadImage(gt_name)
    pred_image = sitk.ReadImage(os.path.join(SEG_DIR, seg_file))

    gt   = sitk.Cast(gt_image == GT_L_GRACILIS, sitk.sitkUInt8)
    gt_arr = sitk.GetArrayFromImage(gt).astype(float)

    pred_arr = sitk.GetArrayFromImage(
        sitk.Cast(pred_image == MUSEG_GRACILIS, sitk.sitkUInt8)).astype(float)
    pred_sitk = sitk.GetImageFromArray(pred_arr.astype(np.uint8))
    pred_sitk.CopyInformation(gt_image)
    pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(gt, pred)

    if gt_arr.sum() > 0 and pred_arr.sum() > 0:
        hd_filter = sitk.HausdorffDistanceImageFilter()
        hd_filter.Execute(gt, pred)
        hd = hd_filter.GetHausdorffDistance()
    else:
        print(f'  empty mask (gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD=NaN')
        hd = np.nan

    results_l_gracilis.append({
        'image':      gt_name,
        'pred_label': seg_file,
        'L_gracilis_lower_dice:':          dice_filter.GetDiceCoefficient(),
        'L_gracilis_lower_Hausdorff:':     hd,
        'L_gracilis_jaccard':              dice_filter.GetJaccardCoefficient(),
        'L_gracilis_volume_similarity':    dice_filter.GetVolumeSimilarity(),
        'L_gracilis_falseNegative':        dice_filter.GetFalseNegativeError(),
        'L_gracilis_falsePostivie':        dice_filter.GetFalsePositiveError(),
        'L_gracilis_binary_cross_entropy': binary_cross_entropy(gt_arr, pred_arr),
        'L_gracilis_boundary_iou_3d':      boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr),
        'L_gracilis_inter_slice_dice_pred': inter_slice_dice(pred_arr),
        'L_gracilis_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
    })

df_l_gracilis = pd.DataFrame(results_l_gracilis)
df_l_gracilis.to_csv(os.path.join(RESULT_DIR, 'df_l_gracilis_museg.csv'))
df_l_gracilis

## Right Sartorius

In [ ]:
results_r_sart = []

for seg_file in seg_files:
    stem    = seg_file.replace('_museg.nii.gz', '')
    subject = stem.split('_FATFRACTION')[0]
    m       = re.search(r'stack(\d+)', stem)
    if not m:
        continue
    stack_num = m.group(1)
    gt_name = os.path.join(GT_BASE, subject, 'SegmentationMasks',
                           f'combined_gt_stack{stack_num}.mha')
    print(seg_file)

    gt_image   = sitk.ReadImage(gt_name)
    pred_image = sitk.ReadImage(os.path.join(SEG_DIR, seg_file))

    gt   = sitk.Cast(gt_image == GT_R_SARTORIUS, sitk.sitkUInt8)
    gt_arr = sitk.GetArrayFromImage(gt).astype(float)

    pred_arr = sitk.GetArrayFromImage(
        sitk.Cast(pred_image == MUSEG_SARTORIUS, sitk.sitkUInt8)).astype(float)
    pred_sitk = sitk.GetImageFromArray(pred_arr.astype(np.uint8))
    pred_sitk.CopyInformation(gt_image)
    pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(gt, pred)

    if gt_arr.sum() > 0 and pred_arr.sum() > 0:
        hd_filter = sitk.HausdorffDistanceImageFilter()
        hd_filter.Execute(gt, pred)
        hd = hd_filter.GetHausdorffDistance()
    else:
        print(f'  empty mask (gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD=NaN')
        hd = np.nan

    results_r_sart.append({
        'image':      gt_name,
        'pred_label': seg_file,
        'R_sart_lower_dice:':          dice_filter.GetDiceCoefficient(),
        'R_sart_lower_Hausdorff:':     hd,
        'R_sart_jaccard':              dice_filter.GetJaccardCoefficient(),
        'R_sart_volume_similarity':    dice_filter.GetVolumeSimilarity(),
        'R_sart_falseNegative':        dice_filter.GetFalseNegativeError(),
        'R_sart_falsePostivie':        dice_filter.GetFalsePositiveError(),
        'R_sart_binary_cross_entropy': binary_cross_entropy(gt_arr, pred_arr),
        'R_sart_boundary_iou_3d':      boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr),
        'R_sart_inter_slice_dice_pred': inter_slice_dice(pred_arr),
        'R_sart_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
    })

df_r_sart = pd.DataFrame(results_r_sart)
df_r_sart.to_csv(os.path.join(RESULT_DIR, 'df_r_sart_museg.csv'))
df_r_sart

## Left Sartorius

In [ ]:
results_l_sart = []

for seg_file in seg_files:
    stem    = seg_file.replace('_museg.nii.gz', '')
    subject = stem.split('_FATFRACTION')[0]
    m       = re.search(r'stack(\d+)', stem)
    if not m:
        continue
    stack_num = m.group(1)
    gt_name = os.path.join(GT_BASE, subject, 'SegmentationMasks',
                           f'combined_gt_stack{stack_num}.mha')
    print(seg_file)

    gt_image   = sitk.ReadImage(gt_name)
    pred_image = sitk.ReadImage(os.path.join(SEG_DIR, seg_file))

    gt   = sitk.Cast(gt_image == GT_L_SARTORIUS, sitk.sitkUInt8)
    gt_arr = sitk.GetArrayFromImage(gt).astype(float)

    pred_arr = sitk.GetArrayFromImage(
        sitk.Cast(pred_image == MUSEG_SARTORIUS, sitk.sitkUInt8)).astype(float)
    pred_sitk = sitk.GetImageFromArray(pred_arr.astype(np.uint8))
    pred_sitk.CopyInformation(gt_image)
    pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(gt, pred)

    if gt_arr.sum() > 0 and pred_arr.sum() > 0:
        hd_filter = sitk.HausdorffDistanceImageFilter()
        hd_filter.Execute(gt, pred)
        hd = hd_filter.GetHausdorffDistance()
    else:
        print(f'  empty mask (gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD=NaN')
        hd = np.nan

    results_l_sart.append({
        'image':      gt_name,
        'pred_label': seg_file,
        'L_sart_lower_dice:':          dice_filter.GetDiceCoefficient(),
        'L_sart_lower_Hausdorff:':     hd,
        'L_sart_jaccard':              dice_filter.GetJaccardCoefficient(),
        'L_sart_volume_similarity':    dice_filter.GetVolumeSimilarity(),
        'L_sart_falseNegative':        dice_filter.GetFalseNegativeError(),
        'L_sart_falsePostivie':        dice_filter.GetFalsePositiveError(),
        'L_sart_binary_cross_entropy': binary_cross_entropy(gt_arr, pred_arr),
        'L_sart_boundary_iou_3d':      boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr),
        'L_sart_inter_slice_dice_pred': inter_slice_dice(pred_arr),
        'L_sart_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
    })

df_l_sart = pd.DataFrame(results_l_sart)
df_l_sart.to_csv(os.path.join(RESULT_DIR, 'df_l_sart_museg.csv'))
df_l_sart